In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv("HDFS_structured.csv")

In [3]:
print("Shape:",df.shape)
df.head()

Shape: (2000, 9)


,LineId,Date,Time,Pid,Level,Component,Content,EventId,EventTemplate
0,1,81109,203615,148,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_38865049064139...,E10,PacketResponder <*> for block blk_<*> terminating
1,2,81109,203807,222,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-6952295868487...,E10,PacketResponder <*> for block blk_<*> terminating
2,3,81109,204005,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...,E6,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,81109,204015,308,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_82291938032499...,E10,PacketResponder <*> for block blk_<*> terminating
4,5,81109,204106,329,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-6670958622368...,E10,PacketResponder <*> for block blk_<*> terminating


In [4]:
df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
df.columns

Index(['lineid', 'date', 'time', 'pid', 'level', 'component', 'content',
       'eventid', 'eventtemplate'],
      dtype='str')

In [5]:
df.isnull().sum()

lineid           0
date             0
time             0
pid              0
level            0
component        0
content          0
eventid          0
eventtemplate    0
dtype: int64

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   lineid         2000 non-null   int64
 1   date           2000 non-null   int64
 2   time           2000 non-null   int64
 3   pid            2000 non-null   int64
 4   level          2000 non-null   str  
 5   component      2000 non-null   str  
 6   content        2000 non-null   str  
 7   eventid        2000 non-null   str  
 8   eventtemplate  2000 non-null   str  
dtypes: int64(4), str(5)
memory usage: 489.2 KB


In [7]:
df.describe(include="all")

,lineid,date,time,pid,level,component,content,eventid,eventtemplate
count,2000.000000,2000.000000,2000.000000,2000.000000,2000,2000,2000,2000,2000
unique,NaN,NaN,NaN,NaN,2,6,2000,14,14
top,NaN,NaN,NaN,NaN,INFO,dfs.FSNamesystem,PacketResponder 1 for block blk_38865049064139...,E6,BLOCK* NameSystem.addStoredBlock: blockMap upd...
freq,NaN,NaN,NaN,NaN,1920,659,1,314,314
mean,1000.500000,81110.367500,107375.609500,7771.287500,NaN,NaN,NaN,NaN,NaN
std,577.494589,0.618575,69780.904034,9053.587141,NaN,NaN,NaN,NaN,NaN
min,1.000000,81109.000000,37.000000,13.000000,NaN,NaN,NaN,NaN,NaN
25%,500.750000,81110.000000,52286.750000,28.000000,NaN,NaN,NaN,NaN,NaN
50%,1000.500000,81110.000000,92538.000000,2883.500000,NaN,NaN,NaN,NaN,NaN
75%,1500.250000,81111.000000,151136.750000,15851.250000,NaN,NaN,NaN,NaN,NaN


In [8]:
print("Before:",len(df))
df=df.drop_duplicates()
print("After:",len(df))

Before: 2000
After: 2000


In [9]:
df["Date"]=(df["Date"].astype(str).str.zfill(6))
df["Time"]=(df["Time"].astype(str).str.zfill(6))
df["timestamp"]=pd.to_datetime(
    df["Date"]+" "+df["Time"],
    format="%y%m%d %H%M%S"
)
df["hour"]=(
    df["timestamp"]
    .dt.hour
)

KeyError: 'Date'

In [ ]:
df[["Date","Time","timestamp","hour"]].head()

In [ ]:
df["Level"].unique()

In [ ]:
df["Level"]=(df["Level"].str.upper())

In [ ]:
df["Level"].value_counts()

In [ ]:
df["content_length"]=(df["Content"].str.len())
df["template_length"]=(df["EventTemplate"].str.len())

In [ ]:
df[["content_length","template_length"]].head()

## Encode Categories using LabelEncoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
level_encoder=LabelEncoder()
component_encoder=LabelEncoder()
event_encoder=LabelEncoder()

In [ ]:
df["level_encoded"]=(level_encoder.fit_transform(df["Level"]))
df["component_encoded"]=(component_encoder.fit_transform(df["Component"]))
df["event_encoded"]=(event_encoder.fit_transform(df["EventId"]))

In [ ]:
df[["Level","level_encoded","Component","component_encoded","EventId","event_encoded"]].head()

In [ ]:
import joblib

joblib.dump(level_encoder,"level_encoder.pkl")
joblib.dump(component_encoder,"component_encoder.pkl")
joblib.dump(event_encoder,"event_encoder.pkl")

## drop unnecessary columns

In [ ]:
model_df=df.drop(["Content","EventTemplate"],axis=1)
model_df.head()

## Saving the processed dataset

In [ ]:
model_df.to_csv(
"processed_logs.csv",
index=False
)